# Denoising Model: Inputs and Predictions

This notebook loads a trained `LitS4DenoisingModel` and the `LitDenoisingDataModule`,
then visualises:
1. Example noisy and clean input pairs from the dataloader
2. Model predictions (denoised signals) alongside the noisy inputs and clean targets
3. Per-sample MSE distribution over the test set

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml

from src.models.model import LitS4DenoisingModel
from src.models.networks import S4DSeq2SeqModel
from src.data.data import LitDenoisingDataModule

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

Set the paths to your checkpoint and config file, and the directory containing your HDF5 data.

In [ ]:
# ── Update these paths ────────────────────────────────────────────────────────
CHECKPOINT_PATH = '../runs/denoising/lightning_logs/<run_id>/checkpoints/last.ckpt'
CONFIG_PATH     = '../runs/denoising/config.yaml'   # saved alongside the checkpoint
DATA_DIR        = '/path/to/data/'                  # directory with HDF5 files
# ─────────────────────────────────────────────────────────────────────────────

## 2. Load the Dataloader

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

data_cfg = config['data']['init_args']

dm = LitDenoisingDataModule(
    inputs       = data_cfg['inputs'],
    data_dir     = DATA_DIR,
    cutoff       = data_cfg.get('cutoff', 8192),
    noise_const  = data_cfg.get('noise_const', 1.0),
    apply_filter = data_cfg.get('apply_filter', False),
    batch_size   = 32,   # smaller batch for visualisation
    num_workers  = 0,
)

test_loader = dm.test_dataloader()
inputs = data_cfg['inputs']
print(f'Test batches : {len(test_loader)}')
print(f'Input channels: {inputs}')

## 3. Load the Model

In [ ]:
model_cfg = config['model']['init_args']
enc_cfg   = model_cfg['encoder']['init_args']

encoder = S4DSeq2SeqModel(
    d_input  = enc_cfg['d_input'],
    d_output = enc_cfg['d_output'],
    d_model  = enc_cfg.get('d_model', 128),
    n_layers = enc_cfg.get('n_layers', 6),
    dropout  = enc_cfg.get('dropout', 0.0),
    prenorm  = enc_cfg.get('prenorm', False),
)

model = LitS4DenoisingModel.load_from_checkpoint(
    CHECKPOINT_PATH,
    encoder=encoder,
)
model = model.to(device).eval()
print('Model loaded successfully')
print(f'  Encoder: {type(encoder).__name__}, '
      f'd_model={enc_cfg.get("d_model", 128)}, '
      f'n_layers={enc_cfg.get("n_layers", 6)}')

## 4. Visualise Example Inputs

Plot a few noisy / clean pairs straight from the dataloader to get a feel for
the noise level before running the model.

In [ ]:
N_EXAMPLES = 4          # number of signals to show
CHANNEL_LABELS = ['I channel', 'Q channel']

x_noisy_batch, x_clean_batch = next(iter(test_loader))
# shapes: (B, cutoff, n_channels)

cutoff = x_noisy_batch.shape[1]
t = np.arange(cutoff)

fig, axes = plt.subplots(
    N_EXAMPLES, 2,
    figsize=(14, 2.5 * N_EXAMPLES),
    sharex=True,
)

for i in range(N_EXAMPLES):
    for ch, label in enumerate(CHANNEL_LABELS):
        ax = axes[i, ch]
        ax.plot(t, x_noisy_batch[i, :, ch].numpy(),
                color='tab:blue', alpha=0.6, lw=0.6, label='Noisy')
        ax.plot(t, x_clean_batch[i, :, ch].numpy(),
                color='tab:orange', lw=1.2, label='Clean')
        ax.set_ylabel('Amplitude (norm.)')
        ax.set_title(f'Sample {i+1} — {label}')
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('Time sample')
axes[-1, 1].set_xlabel('Time sample')
fig.suptitle('Example inputs: noisy vs clean I/Q signals', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Run the Model and Visualise Predictions

Feed the same batch through the denoising model and overlay the prediction on
the noisy input and clean target.

In [ ]:
with torch.no_grad():
    x_pred_batch = model(x_noisy_batch.to(device)).cpu()
# shape: (B, cutoff, n_channels)

fig, axes = plt.subplots(
    N_EXAMPLES, 2,
    figsize=(14, 2.5 * N_EXAMPLES),
    sharex=True,
)

for i in range(N_EXAMPLES):
    for ch, label in enumerate(CHANNEL_LABELS):
        ax = axes[i, ch]
        ax.plot(t, x_noisy_batch[i, :, ch].numpy(),
                color='tab:blue', alpha=0.4, lw=0.6, label='Noisy input')
        ax.plot(t, x_clean_batch[i, :, ch].numpy(),
                color='tab:orange', lw=1.2, label='Clean target')
        ax.plot(t, x_pred_batch[i, :, ch].numpy(),
                color='tab:green', lw=1.2, ls='--', label='Predicted (denoised)')
        ax.set_ylabel('Amplitude (norm.)')
        ax.set_title(f'Sample {i+1} — {label}')
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('Time sample')
axes[-1, 1].set_xlabel('Time sample')
fig.suptitle('Denoising predictions: noisy input / clean target / model output', fontsize=13)
fig.tight_layout()
plt.show()

## 6. Zoom in on a Signal Region

The full 8 k-sample trace is hard to read at once. Here we zoom into a short
window (512 samples by default) where the signal is present.

In [ ]:
WINDOW_START = 0      # change to zoom into a different part of the trace
WINDOW_LEN   = 512
SAMPLE_IDX   = 0      # which sample from the batch to show

sl = slice(WINDOW_START, WINDOW_START + WINDOW_LEN)
t_zoom = np.arange(WINDOW_START, WINDOW_START + WINDOW_LEN)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

for ch, label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    ax.plot(t_zoom, x_noisy_batch[SAMPLE_IDX, sl, ch].numpy(),
            color='tab:blue', alpha=0.5, lw=0.8, label='Noisy input')
    ax.plot(t_zoom, x_clean_batch[SAMPLE_IDX, sl, ch].numpy(),
            color='tab:orange', lw=1.5, label='Clean target')
    ax.plot(t_zoom, x_pred_batch[SAMPLE_IDX, sl, ch].numpy(),
            color='tab:green', lw=1.5, ls='--', label='Predicted')
    ax.set_xlabel('Time sample')
    ax.set_ylabel('Amplitude (norm.)')
    ax.set_title(f'Sample {SAMPLE_IDX+1} — {label} (zoom)')
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 7. Power Spectral Density Comparison

Compare the PSD of the noisy input, clean target, and denoised prediction to
see how well the model suppresses out-of-band noise.

In [ ]:
from scipy import signal as sp_signal

FS = 403e6   # sampling frequency in Hz (from noise model)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ch, label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    for sig, name, color in [
        (x_noisy_batch[SAMPLE_IDX, :, ch].numpy(), 'Noisy input',  'tab:blue'),
        (x_clean_batch[SAMPLE_IDX, :, ch].numpy(), 'Clean target', 'tab:orange'),
        (x_pred_batch[ SAMPLE_IDX, :, ch].numpy(), 'Predicted',    'tab:green'),
    ]:
        f, pxx = sp_signal.periodogram(sig, fs=FS)
        ax.semilogy(f / 1e6, pxx, lw=0.8, label=name, color=color)

    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('PSD [arb/Hz]')
    ax.set_title(f'{label} — PSD')
    ax.legend(fontsize=8)

fig.suptitle('Power Spectral Density: noisy / clean / denoised', fontsize=13)
fig.tight_layout()
plt.show()

## 8. Per-Sample MSE Distribution over the Test Set

Compute the reconstruction MSE for every test sample and plot its distribution
to understand the overall denoising quality.

In [ ]:
mse_list = []

for x_noisy, x_clean in test_loader:
    with torch.no_grad():
        x_pred = model(x_noisy.to(device)).cpu()
    # MSE per sample: mean over (L, C)
    mse = ((x_pred - x_clean) ** 2).mean(dim=(1, 2))  # (B,)
    mse_list.append(mse.numpy())

mse_all = np.concatenate(mse_list)
print(f'Test MSE  mean : {mse_all.mean():.6f}')
print(f'Test MSE  median: {np.median(mse_all):.6f}')
print(f'Test MSE  std  : {mse_all.std():.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mse_all, bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(mse_all.mean(),   color='tab:red',    lw=1.5, ls='--', label=f'Mean = {mse_all.mean():.4f}')
ax.axvline(np.median(mse_all), color='tab:orange', lw=1.5, ls=':',  label=f'Median = {np.median(mse_all):.4f}')
ax.set_xlabel('Per-sample MSE (normalised amplitude)')
ax.set_ylabel('Count')
ax.set_title('Test-set reconstruction MSE distribution')
ax.legend()
fig.tight_layout()
plt.show()